In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/reddit_raw.csv")

df.head()

,type,id,title,text,subreddit,score,num_comments,url,brand,query
0,post,1944bbw,Mo Salah in the most recent Vodafone Egypt Ad,NaN,LiverpoolFC,429,37.0,/r/LiverpoolFC/comments/1944bbw/mo_salah_in_th...,vodafone egypt,vodafone egypt
1,comment,khdmv8a,NaN,Captain Egypt,NaN,183,NaN,NaN,vodafone egypt,vodafone egypt
2,comment,khdkdu0,NaN,Drake is going to do this in a music video in ...,NaN,143,NaN,NaN,vodafone egypt,vodafone egypt
3,comment,khdwa18,NaN,Big Fat Albert fan is our Mo.\n\nhttps://previ...,NaN,68,NaN,NaN,vodafone egypt,vodafone egypt
4,comment,khe3k13,NaN,[deleted],NaN,65,NaN,NaN,vodafone egypt,vodafone egypt


In [4]:
print(df.shape)
df.isnull().sum()

(1907, 10)


type               0
id                 0
title           1807
text              33
subreddit       1807
score              0
num_comments    1807
url             1807
brand              0
query              0
dtype: int64

In [5]:
df["text"] = df["title"].fillna("") + " " + df["text"].fillna("")
df = df.drop(columns=["title"], errors="ignore")
df = df[df["text"].str.strip() != ""]
df = df.drop_duplicates(subset=["text"])

In [6]:
print(df.shape)

(1839, 9)


In [7]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)  # remove links
    text = re.sub(r"[^a-zA-Z\u0600-\u06FF\s]", "", text)  # keep arabic + english
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

# نشيل النصوص القصيرة جدًا
df = df[df["clean_text"].str.len() > 10]

df[["text", "clean_text"]].head()

,text,clean_text
0,Mo Salah in the most recent Vodafone Egypt Ad,mo salah in the most recent vodafone egypt ad
1,Captain Egypt,captain egypt
2,Drake is going to do this in a music video in...,drake is going to do this in a music video in ...
3,Big Fat Albert fan is our Mo.\n\nhttps://prev...,big fat albert fan is our mo
5,The goal bandit.,the goal bandit


In [8]:
sample_df = df[["clean_text"]].sample(200, random_state=42).copy()
sample_df["label"] = ""

sample_df.head()

,clean_text,label
1468,lets do this critical update on easter monday ...,
1143,really hard for me to care about inflation whe...,
1621,i pay rs riyal every month in saudi arabia for...,
314,replace them with ai this is ai,
1050,oh jungejunge klingt bel ich hab sowas mit ike...,


In [10]:
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis")

sample_texts = sample_df["clean_text"].tolist()
auto_results = sentiment_model(sample_texts)

sample_df["label"] = [r["label"].lower() for r in auto_results]

sample_df.head()

d:\Project_Depi\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 9820.51it/s]


,clean_text,label
1468,lets do this critical update on easter monday ...,negative
1143,really hard for me to care about inflation whe...,negative
1621,i pay rs riyal every month in saudi arabia for...,negative
314,replace them with ai this is ai,negative
1050,oh jungejunge klingt bel ich hab sowas mit ike...,negative


In [15]:
sample_df["label"].value_counts()

label
negative    160
positive     40
Name: count, dtype: int64

In [11]:
sample_df.to_csv("../data/labeled/sample_for_labeling.csv", index=False, encoding="utf-8-sig")

print("Saved for labeling")

Saved for labeling
